# 04 · Reusable containers

By default every task execution gets a fresh pod: schedule → pull image → start Python →
import your libraries → run → tear down. That overhead (10-60s, worse for big ML images) is
irrelevant for one 2-hour training task and *dominant* for ten thousand 2-second tasks.

**Reusable containers** keep a pool of warm pods that execute many tasks, concurrently,
while preserving Python process state between executions. They are the v2 successor to
v1 **Actors** — and, for many workloads, the Union-native alternative to reaching for Ray
(that comparison is [06](./06-ray-on-union.ipynb)).

**Learning goals**

1. Configure `ReusePolicy` and reason about its capacity math
2. Load a model **once per pod** and serve many task executions from it
3. Build the production micro-batching pattern (reuse + traces + fan-out)
4. Recognize the high-parallelism failure modes reuse mitigates — and the new ones it introduces

> **Backend note:** reusable containers run on Union backends (your self-managed deployment
> qualifies). The task image must include the `unionai-reuse` package — in the *image*, not
> on your laptop.

In [ ]:
import asyncio
from datetime import timedelta
from typing import Dict, List

import flyte

flyte.init_from_config()

reuse_image = (
    flyte.Image.from_debian_base(name="workshop-reuse", python_version=(3, 12))
    .with_pip_packages("unionai-reuse>=0.1.15", "async-lru==2.3.0")
)

## 1. `ReusePolicy`

Four parameters, two axes:

| Parameter | Axis | Meaning |
|---|---|---|
| `replicas` | infrastructure | How many warm pods (int, or `(min, max)` for autoscaling) |
| `concurrency` | throughput | Task executions **per pod** at once (needs `async def` if > 1) |
| `scaledown_ttl` | lifecycle | Idle time before an *individual* pod is torn down |
| `idle_ttl` | lifecycle | Idle time before the *whole pool* shuts down |

**Total capacity = replicas × concurrency.** Resources are *per replica*, and concurrent
executions share them — size memory for `concurrency` simultaneous items.

In [ ]:
batch_env = flyte.TaskEnvironment(
    name="batch_worker",
    image=reuse_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),                       # autoscale 2..8 warm pods
        concurrency=8,                         # 8 async executions per pod
        scaledown_ttl=timedelta(minutes=5),    # per-pod idle teardown
        idle_ttl=timedelta(minutes=15),        # whole-pool idle shutdown
    ),
)
# Capacity: 2×8=16 concurrent executions warm, bursting to 8×8=64.

# The orchestrator coordinates thousands of children — give it memory, not reuse.
driver_env = flyte.TaskEnvironment(
    name="batch_driver",
    image=reuse_image,
    resources=flyte.Resources(cpu="1", memory="4Gi"),
    depends_on=[batch_env],
)

## 2. Warm state: load once, use many times

Because the Python process survives across executions, module-level state persists. The
disciplined way to exploit that is a cached async initializer — `@alru_cache` guarantees
exactly one load even when concurrent executions race at pod startup:

In [ ]:
from async_lru import alru_cache


@alru_cache(maxsize=1)
async def get_model(model_version: str) -> Dict[str, float]:
    # In real life: torch.load / from_pretrained / open a DB pool — the expensive part.
    print(f"loading model {model_version} (happens once per pod)")
    await asyncio.sleep(2)
    return {"bias": 0.1, "scale": 2.0}


@batch_env.task
async def infer(text: str, model_version: str = "v1") -> float:
    model = await get_model(model_version)          # instant after first call on this pod
    return len(text) * model["scale"] + model["bias"]

In [ ]:
@driver_env.task
async def infer_many(texts: List[str]) -> List[float]:
    results: List[float] = []
    async for r in flyte.map.aio(infer, texts, return_exceptions=True):
        if isinstance(r, Exception):
            raise r
        results.append(r)
    return results


run = await flyte.run.aio(infer_many, texts=[f"document-{i}" for i in range(50)])
print(run.url)
await run.wait.aio()
print(run.outputs()[:5])

Watch the run in the UI: 50 `infer` executions land on a handful of `batch_worker-*` pods.
"loading model" appears once **per pod**, not once per task. Compare pod count with
`kubectl get pods -n <project>-<domain> | grep batch_worker`.

## 3. The micro-batching pattern

The production pattern for 100k-1M+ item workloads combines everything so far:

```
1M items → 1,000 batches of 1,000 → flyte.map over warm pool
each batch: submit phase (traced) → wait phase (traced) → results
```

- **Reuse** removes per-batch cold starts (1,000 batches × 30s = 8 wasted pod-hours otherwise)
- **`@flyte.trace`** checkpoints each phase: a retry never re-submits completed work
- **`return_exceptions=True`** at both levels keeps one bad item/batch from killing the run

In [ ]:
@flyte.trace
async def submit_batch(items: List[int]) -> Dict[int, str]:
    # replace with your service's bulk-submit call
    await asyncio.sleep(0.05)
    return {i: f"job-{i}" for i in items}


@flyte.trace
async def wait_batch(job_ids: Dict[int, str]) -> List[int]:
    # replace with polling your service until completion
    await asyncio.sleep(0.1)
    return [i * 2 for i in job_ids]


from functools import partial


@batch_env.task(retries=3)
async def process_batch(start: int, batch_size: int, total: int) -> List[int]:
    items = list(range(start, min(start + batch_size, total)))
    jobs = await submit_batch(items)        # checkpoint 1
    return await wait_batch(jobs)           # checkpoint 2


@driver_env.task
async def microbatch(total: int = 10_000, batch_size: int = 500) -> int:
    starts = list(range(0, total, batch_size))
    fn = partial(process_batch, batch_size=batch_size, total=total)
    done = 0
    async for r in flyte.map.aio(fn, starts, return_exceptions=True, group_name="batches"):
        if isinstance(r, Exception):
            print(f"batch failed after retries: {r!r}")
            continue
        done += len(r)
    return done

> Scaled to 1M items: bump `total`, raise `batch_size` to 1,000-10,000, and widen
> `replicas`. Larger batches = fewer actions and coarser checkpoints; smaller batches =
> finer recovery and better load spread. Start at 1,000 and tune with real timings.

## 4. High-parallelism failure modes

The bottlenecks your team listed map directly onto this pattern:

| Failure mode | Without reuse | With reuse |
|---|---|---|
| **Image-pull throttling** — hundreds of pods pulling a multi-GB image stampede Artifact Registry / node disk | Frequent, at every scale-out | Pulls happen only when the pool grows |
| **Queueing delays** — 10k pods hit scheduler + autoscaler limits | Constant churn | Pool holds capacity; scheduler sees ~10 pods, not 10k |
| **Slow aborts** — cancelling a run must reap thousands of pods | Minutes of teardown | Abort marks in-flight executions; pods just go idle |
| **OOM patterns** | Per-pod, isolated | **Shared pod**: one hungry item can OOM-kill neighbors mid-flight — size memory for `concurrency` × worst item, or lower `concurrency` |
| **Cold-start cost** | Paid per task | Paid per pool scale-up |

The OOM row is the trade-off to respect: reuse shares blast radius. CPU-bound or
memory-spiky work wants `concurrency=1` (still keeps the warm-start win); I/O-bound work
can push concurrency much higher.

## 5. Coming from v1 Actors

| v1 (`union.actor`) | v2 |
|---|---|
| `ActorEnvironment(replica_count=2, ttl_seconds=300)` | `TaskEnvironment(reusable=ReusePolicy(replicas=2, idle_ttl=timedelta(seconds=300)))` |
| `@actor.task` | `@env.task` (same decorator as everything else) |
| `@actor_cache` for warm state | `@alru_cache` / `@lru_cache` on an initializer |
| Fixed `replica_count` | `replicas=(min, max)` autoscaling |
| TTL max 15 min | Configurable `scaledown_ttl` + `idle_ttl` |

## Further reading

- Next: [06-ray-on-union](./06-ray-on-union.ipynb) — when this pattern replaces Ray, and when it doesn't

> 💬 **Discuss:** sort your highest-volume workloads into I/O-bound vs memory-spiky. What `concurrency` and per-replica memory would you start each one at?